# Exploración de los originales bancarios en ADLS

## Objetivo

Comprender la estructura física y lógica de los datos obtenidos mediante
Enable Banking antes de diseñar su carga en tablas Delta de Bronze.

Esta exploración analiza los datos de Santander y Laboral Kutxa almacenados en:

`abfss://raw@graxianodatalake.dfs.core.windows.net/personal/finanzas/openbanking/enablebanking/`

## Preguntas de investigación

1. ¿Cómo se organizan los archivos por extracción, banco y cuenta?
2. ¿Qué contienen los movimientos, saldos, manifiestos y puntos de control?
3. ¿Cómo se representa la paginación, incluidas las páginas vacías?
4. ¿Cómo identificamos una extracción completa, parcial o fallida?
5. ¿Qué repeticiones introduce el solapamiento de las cargas incrementales?
6. ¿Qué información necesitamos conservar para garantizar trazabilidad
   y permitir reprocesamientos?

## Hipótesis iniciales

Estas hipótesis se contrastarán con el código de ingesta y los archivos:

- Cada ejecución genera una carpeta identificada por fecha y `run_id`.
- Los movimientos se guardan en páginas y los saldos en archivos separados.
- Una página vacía no implica necesariamente ausencia de movimientos
  en el resto de la extracción.
- Los manifiestos permiten evaluar el resultado de las extracciones.
- Las cargas con ventanas temporales solapadas pueden repetir movimientos.
- Los archivos de `_state` controlan la ingesta desde la API y no representan
  datos de negocio.

## Método

Para cada etapa del análisis:

1. Formular una pregunta concreta.
2. Ejecutar una inspección de solo lectura.
3. Examinar el resultado.
4. Registrar observaciones, conclusiones y dudas pendientes.

Se priorizarán esquemas, recuentos y metadatos, evitando mostrar información
bancaria personal cuando no sea necesaria.

## Alcance

En este notebook no se crearán ni modificarán tablas, archivos originales
o puntos de control. La estrategia de carga se definirá al finalizar
la exploración.

## 1. Acceso y organización del almacenamiento

### Pregunta

¿Podemos acceder a los originales bancarios y qué carpetas existen
en su directorio raíz?

### Comprobación

Listaremos el directorio de Enable Banking. Este resultado permitirá
distinguir las carpetas de extracciones de las carpetas internas de control.

In [0]:
base_path = (
    "abfss://raw@graxianodatalake.dfs.core.windows.net/"
    "personal/finanzas/openbanking/enablebanking/"
)

display(dbutils.fs.ls(base_path))

### Resultados y conclusiones — Organización del directorio raíz

**Observaciones**

- El acceso al directorio de Enable Banking funciona.
- Hay 18 carpetas de extracción, desde el 2 hasta el 19 de noviembre de 2026,
  sin huecos entre esas fechas.
- Las carpetas siguen la convención `extraction_date=AAAA-MM-DD`.
- Existe un directorio separado llamado `_state`.

**Interpretación**

La organización permite localizar los originales por fecha de extracción.
Esta fecha no debe confundirse con la fecha de los movimientos bancarios.

La presencia de carpetas en días consecutivos es compatible con una ingesta
diaria, pero no demuestra que todas las ejecuciones hayan terminado
correctamente ni que contengan datos de ambos bancos.

**Limitaciones**

- El listado todavía no muestra las ejecuciones, cuentas o archivos interiores.
- El tamaño cero corresponde a las entradas de directorio; no significa
  que sus archivos estén vacíos.
- Falta inspeccionar `_state` y los manifiestos para confirmar su función
  y el estado de las cargas.

**Siguiente comprobación**

Explorar una fecha concreta para identificar las ejecuciones y la
organización por banco y cuenta.

## 2. Estructura de una extracción

### Pregunta

¿Cómo se organizan las ejecuciones, bancos, cuentas y archivos dentro
de una fecha de extracción?

### Método

Inspeccionaremos el 2 de noviembre, la primera fecha disponible.
Listaremos sus archivos y rutas relativas sin leer su contenido.

La estructura observada en esta fecha se contrastará después con
una extracción más reciente.

In [0]:
inspection_path = base_path + "extraction_date=2026-11-02/"


def list_files_recursive(path):
    files = []

    for item in dbutils.fs.ls(path):
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append((
                item.path.removeprefix(base_path),
                item.size,
            ))

    return files


inventory = list_files_recursive(inspection_path)

inventory_df = spark.createDataFrame(
    inventory,
    schema="relative_path STRING, size_bytes LONG",
)

display(inventory_df.orderBy("relative_path"))

### Resultados y conclusiones — Estructura de una extracción

**Observaciones**

El 2 de noviembre contiene tres ejecuciones distintas:

| Ejecución | Bancos presentes | Páginas de movimientos | Archivos de saldos |
|---|---|---:|---:|
| `012628…_af43e26cf7cd` | Laboral y Santander | 11 | 2 |
| `020156…_9d27c1c7213a` | Santander | 1 | 1 |
| `020200…_c45fe76ca2f0` | Laboral | 1 | 1 |

Cada ejecución incluye un `manifest.json`.
Los datos se organizan por banco y cuenta, separando saldos y movimientos.

**Contexto de las pruebas**

Durante el primer día se ejecutó la ingesta varias veces para probarla.
Las tres ejecuciones no representan necesariamente información nueva
ni conjuntos de movimientos distintos.

**Conclusiones para el diseño**

- `extraction_date` identifica la fecha de extracción; `run_id` distingue
  las ejecuciones realizadas dentro de esa fecha.
- No podemos asumir una única ejecución por día.
- No debemos seleccionar únicamente la última ejecución del día:
  en este ejemplo contiene solo Laboral y dejaría fuera Santander.
- `account=001` es una posición dentro de la extracción, no una clave
  bancaria global. Debemos contrastarla con los identificadores del JSON.
- El tamaño del archivo no permite determinar por sí solo cuántos
  movimientos contiene ni si la extracción terminó correctamente.

**Pendiente de validar**

Revisar los tres manifiestos para conocer el estado, la estrategia,
los rangos solicitados y los recuentos de cada ejecución.

La selección de ejecuciones y la deduplicación se definirán después
de esta comprobación.

## 3. Manifiestos y resultado de las ejecuciones

### Pregunta

¿Qué diferencias existen entre las tres ejecuciones del primer día
y cómo documentan su resultado?

### Método

Leer los tres manifiestos, inspeccionar su esquema y comparar sus
campos de control. No se modificarán los archivos originales.

In [0]:
manifest_paths = [
    base_path + relative_path
    for relative_path, _ in inventory
    if relative_path.endswith("/manifest.json")
]

manifests_df = (
    spark.read
    .option("multiLine", "true")
    .json(manifest_paths)
)

manifests_df.printSchema()

In [0]:
from pyspark.sql import functions as F

# Resumen por ejecución.
display(
    manifests_df.select(
        "run_id",
        "started_at",
        "finished_at",
        F.col("status").alias("run_status"),
        F.col("request.strategy").alias("strategy"),
        F.col("request.date_from").alias("date_from"),
        F.col("request.date_to").alias("date_to"),
        F.size("results").alias("numero_resultados"),
    )
    .orderBy("started_at")
)

# Detalle por banco, cuenta y endpoint.
display(
    manifests_df
    .select(
        "run_id",
        F.col("status").alias("run_status"),
        F.explode_outer("results").alias("result"),
    )
    .select(
        "run_id",
        "run_status",
        "result.bank",
        "result.account_index",
        "result.endpoint",
        F.col("result.status").alias("endpoint_status"),
        F.size("result.files").alias("archivos_guardados"),
        "result.pages_saved",
        "result.record_count",
        "result.records_saved",
    )
    .orderBy("run_id", "bank", "account_index", "endpoint")
)

### Conclusiones — Manifiestos del primer día

**Resultados observados**

| Ejecución | Banco | Páginas de movimientos | Movimientos declarados | Registros de saldos |
|---|---|---:|---:|---:|
| Histórica · `longest` | Laboral | 5 | 43 | 2 |
| Histórica · `longest` | Santander | 6 | 144 | 2 |
| Prueba · `default` | Santander | 1 | 11 | 2 |
| Prueba · `default` | Laboral | 1 | 4 | 2 |

Los ocho resultados por banco y endpoint declaran `success`.

**Conclusiones**

- La ejecución histórica declara 187 movimientos entre ambos bancos.
- Las pruebas posteriores declaran 15 movimientos adicionales en sus
  respuestas; no podemos considerarlos movimientos nuevos sin compararlos.
- Los recuentos de archivos del manifiesto coinciden con el inventario
  observado: 11 páginas históricas y 2 páginas de pruebas.
- El número de páginas no equivale al número de páginas con movimientos:
  anteriormente observamos páginas vacías con continuación.
- Cada respuesta de saldos contiene dos registros. No implica dos cuentas;
  debemos inspeccionar qué tipos de saldo representan.

**Implicaciones para el diseño**

Conservaremos la distinción entre ejecución, banco, cuenta y endpoint.
La deduplicación de movimientos no debe basarse únicamente en `run_id`
ni en la fecha de extracción.

**Limitaciones**

La coherencia entre manifiesto e inventario no demuestra por sí sola
la integridad del histórico bancario. Queda pendiente contrastar los
recuentos declarados con el contenido de las páginas.

## 4. Comparación con una extracción reciente

### Pregunta

¿Mantienen las cargas posteriores la estructura del primer día y cómo
registran su estrategia, intervalo y resultado?

### Método

Seleccionaremos la carpeta con la fecha de extracción más reciente
disponible y examinaremos sus archivos y manifiestos.

No asumiremos que contiene una sola ejecución ni que terminó correctamente.

In [0]:
date_folders = [
    item
    for item in dbutils.fs.ls(base_path)
    if item.isDir() and item.name.startswith("extraction_date=")
]

latest_folder = max(date_folders, key=lambda item: item.name)
latest_inventory = list_files_recursive(latest_folder.path)

print("Fecha inspeccionada:", latest_folder.name)

display(
    spark.createDataFrame(
        latest_inventory,
        schema="relative_path STRING, size_bytes LONG",
    ).orderBy("relative_path")
)

latest_manifest_paths = [
    base_path + relative_path
    for relative_path, _ in latest_inventory
    if relative_path.endswith("/manifest.json")
]

latest_manifests_df = (
    spark.read
    .option("multiLine", "true")
    .json(latest_manifest_paths)
)

display(
    latest_manifests_df.select(
        "run_id",
        "started_at",
        F.col("status").alias("run_status"),
        F.col("request.strategy").alias("strategy"),
        F.col("request.date_from").alias("date_from"),
        F.col("request.date_to").alias("date_to"),
        F.size("results").alias("numero_resultados"),
    ).orderBy("started_at")
)

### Resultados y conclusiones — Histórico inicial y cargas incrementales

**Estrategia de extracción**

La primera carga utilizó `longest` para solicitar todo el histórico que
permitía la API y establecer una base inicial.

Después se programaron extracciones con estrategia `default` y un rango
explícito de fechas, evitando solicitar todo el histórico en cada ejecución.

**Evidencia de la extracción reciente**

El 19 de noviembre presenta dos ejecuciones independientes:

| Banco | Estrategia | Desde | Hasta | Estado global |
|---|---|---|---|---|
| Santander | `default` | 2026-11-11 | 2026-11-19 | `success` |
| Laboral | `default` | 2026-11-11 | 2026-11-19 | `success` |

Cada ejecución contiene un manifiesto, un archivo de saldos y una página
de movimientos.

**Conclusiones**

- El histórico inicial y las extracciones posteriores deben considerarse
  conjuntamente para construir el histórico analítico.
- Una fecha de extracción puede contener varias ejecuciones, también
  durante la operación programada.
- Las cargas incrementales consultan intervalos que pueden solaparse.
  Por tanto, una respuesta diaria no contiene necesariamente solo
  movimientos nuevos.
- La fecha de extracción identifica cuándo se obtuvo la respuesta,
  no cuándo ocurrió cada movimiento.
- El rango observado es del 11 al 19 de noviembre. Aún debemos contrastar
  con los puntos de control cómo se calculó su límite inferior.

**Implicación para Bronze y Silver**

Bronze debe conservar las respuestas y su trazabilidad sin descartar
extracciones por compartir fecha.

Silver deberá identificar movimientos repetidos y posibles cambios
de estado o contenido entre extracciones. La clave y las reglas de
actualización todavía están pendientes de definir.

## 5. Estructura de los saldos

### Pregunta

¿Qué estructura tiene una respuesta de saldos y qué representan
los registros incluidos para una cuenta?

### Método

Inspeccionar primero el esquema de un archivo reciente de saldos,
sin mostrar todavía identificadores de cuenta ni importes.

In [0]:
balance_path = next(
    base_path + relative_path
    for relative_path, _ in latest_inventory
    if relative_path.endswith("/balances.json")
)

balances_df = (
    spark.read
    .option("multiLine", "true")
    .json(balance_path)
)

balances_df.printSchema()

In [0]:
balance_paths = [
    base_path + relative_path
    for relative_path, _ in latest_inventory
    if relative_path.endswith("/balances.json")
]

balances_df = (
    spark.read
    .option("multiLine", "true")
    .json(balance_paths)
)

display(
    balances_df
    .select(
        "bank",
        F.explode("response.balances").alias("balance"),
    )
    .select(
        "bank",
        "balance.balance_type",
        "balance.name",
        F.col("balance.balance_amount.currency").alias("currency"),
        "balance.reference_date",
    )
    .orderBy("bank", "balance_type")
)

### Observaciones sobre el esquema de saldos

- Cada archivo identifica el banco, la cuenta, la ejecución y el momento
  de extracción.
- `response.balances` contiene varios registros de saldo por cuenta.
- Cada registro incluye tipo, nombre, importe, moneda y referencias temporales.
- El importe se representa como texto en el original.
- `page_number` aparece como texto en el esquema inferido. No asumiremos
  que representa paginación en este endpoint: debemos comprobar su valor.

**Pendiente de interpretación**

Identificar los tipos de saldo presentes antes de definir cómo se
representarán en las tablas analíticas. No deben sumarse los registros
sin conocer su significado.

### Resultados y conclusiones — Tipos de saldo

**Valores observados**

| Banco | Código | Nombre devuelto por la API | Moneda | Fecha de referencia |
|---|---|---|---|---|
| Laboral | CLBD | Accounting balance | EUR | No informada |
| Laboral | ITAV | Interim available balance | EUR | No informada |
| Santander | CLBD | Accounting balance | EUR | 2026-11-20 |
| Santander | OPBD | Opening booked balance | EUR | 2026-11-20 |

**Interpretación**

Los registros expresan distintas perspectivas del saldo: contable,
disponible o contable de apertura, según el nombre proporcionado por la API.

No son importes que deban sumarse entre sí.

**Conclusiones para el modelado**

- Conservar `balance_type` y `name` para distinguir el significado
  de cada registro.
- No asumir que ambos bancos proporcionan los mismos tipos de saldo.
- Mantener los valores nulos de `reference_date`: no equivalen
  a la fecha de extracción.
- Separar `extracted_at`, fecha de extracción y fecha de referencia
  del saldo.
- En Silver, convertir el importe a un tipo decimal y conservar
  su moneda.
- Para cualquier comparación entre bancos, definir explícitamente
  qué tipo de saldo se utilizará.

**Observación temporal**

La ruta de extracción es del 19 de noviembre, mientras Santander
informa una fecha de referencia del día 20.

La ejecución comenzó alrededor de las 23:00 UTC del día 19,
equivalente a las 00:00 del día 20 en Madrid. Es compatible con
la diferencia observada, aunque no demuestra cómo asigna el banco
su fecha de referencia.

In [0]:
display(
    balances_df.select(
        "bank",
        "endpoint",
        "page_number",
        "extracted_at",
    ).orderBy("bank")
)

### Comprobación adicional — Paginación y momento de extracción

**Observaciones**

- Ambos archivos tienen `endpoint = balances`.
- `page_number` es nulo en ambos bancos.
- `extracted_at` incluye hora y zona horaria explícita (`+00:00`, UTC).
- Las respuestas se obtuvieron con pocos segundos de diferencia.

**Conclusiones**

En estos archivos, `page_number` no aporta información de paginación.
Conservaremos su valor nulo, sin sustituirlo por cero ni por uno.

El tipo `string` observado procede de la inferencia de esquema sobre
valores nulos; no permite concluir que ese sea el tipo lógico del campo.

Los saldos son observaciones obtenidas en un momento concreto.
`extracted_at` registra cuándo se consultaron y debe mantenerse separado
de `reference_date`, que puede estar ausente.

**Alcance de la evidencia**

Esta comprobación corresponde a los dos archivos recientes inspeccionados.
No demuestra que todos los archivos históricos tengan idéntica estructura.

## 6. Puntos de control de la ingesta incremental

### Pregunta

¿Cómo se organizan los puntos de control que utiliza la extracción
para recordar hasta qué fecha ha consultado cada cuenta?

### Método

Listar los archivos de `_state` sin leer su contenido.
Después seleccionaremos un punto de control para inspeccionar su estructura.

Estos archivos se tratarán como información operativa,
separada de los movimientos y saldos.

In [0]:
state_inventory = list_files_recursive(base_path + "_state/")

display(
    spark.createDataFrame(
        state_inventory,
        schema="relative_path STRING, size_bytes LONG",
    ).orderBy("relative_path")
)

### Resultados parciales — Organización de los puntos de control

**Observaciones**

- Los archivos siguen la estructura:
  `_state/transactions/{banco}/{clave_cuenta}/{run_id}.json`.
- Hay una clave de cuenta distinta para cada banco.
- El listado contiene 19 archivos por banco: dos del primer día
  y uno por cada fecha posterior, del 3 al 19 de noviembre.
- Los nombres coinciden con identificadores de ejecuciones observadas
  en las carpetas de extracción.

**Interpretación**

La estructura conserva varios puntos de control históricos por cuenta,
en lugar de mostrar un único archivo sobrescrito.

**Pendiente de comprobar**

El listado no permite determinar hasta qué fecha cubre cada punto
de control, cómo se selecciona el vigente ni cuándo se guarda.
Inspeccionaremos su contenido y lo contrastaremos con el código.

In [0]:
state_paths = [
    base_path + relative_path
    for relative_path, _ in state_inventory
    if relative_path.endswith(".json")
]

state_df = (
    spark.read
    .option("multiLine", "true")
    .json(state_paths)
)

display(
    state_df.select(
        "bank",
        "run_id",
        "mode",
        "covered_through",
        "completed_at",
    ).orderBy("bank", "completed_at")
)

### Resultados y conclusiones — Evolución de los puntos de control

**Observaciones**

- Ambos bancos tienen un punto de control inicial con `mode = initial`
  y `covered_through = 2026-11-02`.
- Las ejecuciones posteriores figuran como `incremental`.
- Las pruebas del primer día mantienen la misma fecha cubierta.
- Desde el 3 hasta el 19 de noviembre, `covered_through` avanza diariamente
  en ambos bancos, sin huecos en las fechas observadas.
- Cada punto conserva el `run_id` y el momento de finalización en UTC.

**Interpretación**

`covered_through` representa la fecha hasta la que el proceso declara
haber completado la consulta. No es necesariamente la fecha del último
movimiento recibido: una consulta puede completarse sin devolver
movimientos nuevos.

Los puntos de control documentan la extracción desde la API.
No indican que esos archivos ya estén cargados en Bronze o Silver.

**Relación con el solapamiento**

Antes de la extracción del 19 de noviembre, el último punto observado
cubría el día 18.

El manifiesto del día 19 solicita datos desde el 11 hasta el 19.
El inicio coincide con restar siete días al punto de control anterior:

`2026-11-18 − 7 días = 2026-11-11`

Este resultado es compatible con una ventana incremental solapada.
La regla exacta se contrastará con el código de extracción.

**Implicaciones para la carga analítica**

- Mantener separados los controles de extracción API → RAW
  y los controles de carga RAW → Bronze.
- No utilizar `_state` como si contuviera movimientos.
- No modificar los puntos de control de la API durante esta exploración.
- Conservar la trazabilidad de las distintas extracciones, aunque
  sus intervalos se solapen.

**Limitaciones**

El avance del punto de control no demuestra por sí solo que el banco
haya devuelto todos los movimientos esperados. Aún falta contrastar
los manifiestos con el contenido de los archivos.

## 7. Integridad de las páginas de movimientos

### Objetivo
Contrastar el número de archivos y movimientos guardados con los
recuentos declarados en los manifiestos del 2 de noviembre.

Ese día contiene la extracción histórica inicial y dos ejecuciones
incrementales, lo que permite comprobar ambos modos de extracción.

### Comprobaciones
- Cada ejecución y cuenta tiene las páginas que declara su manifiesto.
- La suma de movimientos de las páginas coincide con `records_saved`.
- Las páginas vacías se cuentan como archivos válidos.
- Una respuesta sin el array `transactions` se considera una anomalía.

Los recuentos representan movimientos recibidos en cada extracción,
no movimientos únicos entre ejecuciones.

In [0]:
from pyspark.sql import functions as F

# Archivos de movimientos del primer día; excluimos los checkpoints.
transaction_paths = [
    base_path + relative_path
    for relative_path, _ in inventory
    if "/transactions/page_" in relative_path
    and relative_path.endswith(".json")
]

# Una fila por página JSON.
pages_df = (
    spark.read
    .option("multiLine", "true")
    .json(transaction_paths)
    .select(
        "run_id",
        "bank",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"/account=(\d+)/",
            1,
        ).cast("long").alias("account_index"),
        F.col("_metadata.file_path").alias("source_path"),
        F.col("response.transactions").alias("transactions"),
    )
)

# Recuento observado en los archivos.
observed_df = (
    pages_df
    .groupBy("run_id", "bank", "account_index")
    .agg(
        F.countDistinct("source_path").alias("archivos_observados"),
        F.sum(
            F.when(
                F.col("transactions").isNotNull(),
                F.size("transactions"),
            )
        ).alias("movimientos_observados"),
        F.sum(
            F.when(F.col("transactions").isNull(), 1).otherwise(0)
        ).alias("paginas_sin_array"),
    )
)

# Recuento declarado por cada resultado de tipo transactions.
declared_df = (
    manifests_df
    .select("run_id", F.explode("results").alias("result"))
    .where(F.col("result.endpoint") == "transactions")
    .select(
        "run_id",
        F.col("result.bank").alias("bank"),
        F.col("result.account_index").alias("account_index"),
        F.col("result.status").alias("endpoint_status"),
        F.size("result.files").alias("archivos_declarados"),
        F.col("result.pages_saved").alias("paginas_declaradas"),
        F.col("result.records_saved").alias("movimientos_declarados"),
    )
)

comparison_df = (
    declared_df
    .join(
        observed_df,
        ["run_id", "bank", "account_index"],
        "full",
    )
    .withColumn(
        "resultado",
        F.when(
            (F.col("endpoint_status") == "success")
            & (
                F.col("archivos_declarados")
                == F.col("archivos_observados")
            )
            & (
                F.col("paginas_declaradas")
                == F.col("archivos_observados")
            )
            & (
                F.col("movimientos_declarados")
                == F.col("movimientos_observados")
            )
            & (F.col("paginas_sin_array") == 0),
            F.lit("OK"),
        ).otherwise(F.lit("REVISAR")),
    )
)

display(comparison_df.orderBy("run_id", "bank", "account_index"))

### Conclusiones

Las cuatro combinaciones de ejecución, banco y cuenta analizadas
presentan coherencia entre los manifiestos y los archivos leídos:

- Histórico Laboral: 5 páginas y 43 movimientos.
- Histórico Santander: 6 páginas y 144 movimientos.
- Incremental Santander: 1 página y 11 movimientos.
- Incremental Laboral: 1 página y 4 movimientos.

Todos los archivos contienen el array `transactions`. Las páginas
vacías del histórico son compatibles con los recuentos declarados.

Esta comprobación valida los recuentos del 2 de noviembre.
No demuestra que la API haya entregado todo el histórico disponible,
ni que los movimientos sean únicos entre ejecuciones. Tampoco
comprueba todavía que las rutas coincidan exactamente con las
enumeradas en cada manifiesto.

Por tanto, conservaremos todas las extracciones como evidencia.
La identificación de movimientos repetidos se estudiará antes
de definir la deduplicación en Silver.

## 8. Validación de todas las extracciones de movimientos

### Objetivo
Comprobar que los recuentos de páginas y movimientos coinciden con
los manifiestos en todas las fechas disponibles.

### Criterios
Una combinación de ejecución, banco y cuenta será correcta si:
- El resultado del endpoint es `success`.
- Los archivos observados coinciden con los archivos y páginas declarados.
- Los movimientos observados coinciden con `records_saved`.
- Todas las páginas contienen el array `transactions`.

La validación comprueba recuentos, no la unicidad de movimientos
ni la correspondencia exacta de las rutas declaradas.

In [0]:
# Inventariamos únicamente las carpetas de extracción.
extraction_folders = [
    item
    for item in dbutils.fs.ls(base_path)
    if item.isDir() and item.name.startswith("extraction_date=")
]

all_inventory = []
for folder in extraction_folders:
    all_inventory.extend(list_files_recursive(folder.path))

all_manifest_paths = [
    base_path + relative_path
    for relative_path, _ in all_inventory
    if relative_path.endswith("/manifest.json")
]

all_transaction_paths = [
    base_path + relative_path
    for relative_path, _ in all_inventory
    if "/transactions/page_" in relative_path
    and relative_path.endswith(".json")
]

all_manifests_df = (
    spark.read
    .option("multiLine", "true")
    .json(all_manifest_paths)
)

all_pages_df = (
    spark.read
    .option("multiLine", "true")
    .json(all_transaction_paths)
    .select(
        "run_id",
        "bank",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"/account=(\d+)/",
            1,
        ).cast("long").alias("account_index"),
        F.col("_metadata.file_path").alias("source_path"),
        F.col("response.transactions").alias("transactions"),
    )
)

all_observed_df = (
    all_pages_df
    .groupBy("run_id", "bank", "account_index")
    .agg(
        F.countDistinct("source_path").alias("archivos_observados"),
        F.sum(
            F.when(
                F.col("transactions").isNotNull(),
                F.size("transactions"),
            )
        ).alias("movimientos_observados"),
        F.sum(
            F.when(F.col("transactions").isNull(), 1).otherwise(0)
        ).alias("paginas_sin_array"),
    )
)

all_declared_df = (
    all_manifests_df
    .select("run_id", F.explode("results").alias("result"))
    .where(F.col("result.endpoint") == "transactions")
    .select(
        "run_id",
        F.col("result.bank").alias("bank"),
        F.col("result.account_index").alias("account_index"),
        F.col("result.status").alias("endpoint_status"),
        F.size("result.files").alias("archivos_declarados"),
        F.col("result.pages_saved").alias("paginas_declaradas"),
        F.col("result.records_saved").alias("movimientos_declarados"),
    )
)

all_comparison_df = (
    all_declared_df
    .join(
        all_observed_df,
        ["run_id", "bank", "account_index"],
        "full",
    )
    .withColumn(
        "resultado",
        F.when(
            (F.col("endpoint_status") == "success")
            & (
                F.col("archivos_declarados")
                == F.col("archivos_observados")
            )
            & (
                F.col("paginas_declaradas")
                == F.col("archivos_observados")
            )
            & (
                F.col("movimientos_declarados")
                == F.col("movimientos_observados")
            )
            & (F.col("paginas_sin_array") == 0),
            F.lit("OK"),
        ).otherwise(F.lit("REVISAR")),
    )
)

# Resumen por banco.
display(
    all_comparison_df
    .groupBy("bank", "resultado")
    .agg(
        F.count("*").alias("extracciones"),
        F.sum("archivos_observados").alias("paginas"),
        F.sum("movimientos_observados").alias(
            "movimientos_con_repeticiones"
        ),
    )
    .orderBy("bank", "resultado")
)

# Detalle de posibles discrepancias.
display(
    all_comparison_df
    .where(F.col("resultado") == "REVISAR")
    .orderBy("run_id", "bank", "account_index")
)

### Conclusiones

| Banco | Extracciones validadas | Páginas | Movimientos recibidos |
|---|---:|---:|---:|
| Laboral | 19 | 23 | 78 |
| Santander | 19 | 24 | 437 |
| Total | 38 | 47 | 515 |

Todas las extracciones analizadas cumplen los controles:
- Estado del endpoint `success`.
- Coincidencia entre archivos observados y archivos/páginas declarados.
- Coincidencia entre movimientos observados y declarados.
- Ninguna página sin el array `transactions`.

Por banco, las 19 extracciones corresponden al histórico inicial,
una ejecución incremental adicional el primer día y las 17 ejecuciones
diarias del 3 al 19 de noviembre.

Los 515 movimientos son observaciones acumuladas, no movimientos únicos:
las consultas incrementales utilizan ventanas de fechas solapadas.

No se han detectado discrepancias de recuento. Esto no demuestra
la integridad del histórico proporcionado por el banco ni valida
la correspondencia exacta de las rutas enumeradas en los manifiestos.

## 9. Validación de las extracciones de saldos

### Objetivo
Contrastar los archivos y registros de saldos guardados con los
recuentos declarados en los manifiestos.

### Criterios
Una combinación de ejecución, banco y cuenta será correcta si:
- El estado del endpoint es `success`.
- Los archivos observados coinciden con los declarados.
- Los registros observados coinciden con `record_count`.
- Todos los archivos contienen el array `balances`.

Cada registro representa un tipo de saldo observado en una extracción.
No se suman sus importes: los distintos tipos de saldo tienen
significados diferentes.

In [0]:
all_balance_paths = [
    base_path + relative_path
    for relative_path, _ in all_inventory
    if relative_path.endswith("/balances.json")
]

all_balances_df = (
    spark.read
    .option("multiLine", "true")
    .json(all_balance_paths)
    .select(
        "run_id",
        "bank",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"/account=(\d+)/",
            1,
        ).cast("long").alias("account_index"),
        F.col("_metadata.file_path").alias("source_path"),
        F.col("response.balances").alias("balances"),
    )
)

balances_observed_df = (
    all_balances_df
    .groupBy("run_id", "bank", "account_index")
    .agg(
        F.countDistinct("source_path").alias("archivos_observados"),
        F.sum(
            F.when(
                F.col("balances").isNotNull(),
                F.size("balances"),
            )
        ).alias("saldos_observados"),
        F.sum(
            F.when(F.col("balances").isNull(), 1).otherwise(0)
        ).alias("archivos_sin_array"),
    )
)

balances_declared_df = (
    all_manifests_df
    .select("run_id", F.explode("results").alias("result"))
    .where(F.col("result.endpoint") == "balances")
    .select(
        "run_id",
        F.col("result.bank").alias("bank"),
        F.col("result.account_index").alias("account_index"),
        F.col("result.status").alias("endpoint_status"),
        F.size("result.files").alias("archivos_declarados"),
        F.col("result.record_count").alias("saldos_declarados"),
    )
)

balances_comparison_df = (
    balances_declared_df
    .join(
        balances_observed_df,
        ["run_id", "bank", "account_index"],
        "full",
    )
    .withColumn(
        "resultado",
        F.when(
            (F.col("endpoint_status") == "success")
            & (
                F.col("archivos_declarados")
                == F.col("archivos_observados")
            )
            & (
                F.col("saldos_declarados")
                == F.col("saldos_observados")
            )
            & (F.col("archivos_sin_array") == 0),
            F.lit("OK"),
        ).otherwise(F.lit("REVISAR")),
    )
)

# Resumen por banco.
display(
    balances_comparison_df
    .groupBy("bank", "resultado")
    .agg(
        F.count("*").alias("extracciones"),
        F.sum("archivos_observados").alias("archivos"),
        F.sum("saldos_observados").alias("registros_de_saldo"),
    )
    .orderBy("bank", "resultado")
)

# Posibles discrepancias.
display(
    balances_comparison_df
    .where(F.col("resultado") == "REVISAR")
    .orderBy("run_id", "bank", "account_index")
)

### Conclusiones

| Banco | Extracciones validadas | Archivos | Registros de saldo |
|---|---:|---:|---:|
| Laboral | 19 | 19 | 38 |
| Santander | 19 | 19 | 38 |
| Total | 38 | 38 | 76 |

Todas las extracciones cumplen los controles:
- Estado del endpoint `success`.
- Coincidencia entre archivos observados y declarados.
- Coincidencia entre registros observados y `record_count`.
- Ningún archivo sin el array `balances`.

Los 76 registros representan observaciones de saldo en distintas
extracciones. No representan cuentas distintas ni deben sumarse
sus importes entre tipos o fechas.

Este control valida los recuentos. No comprueba todavía la validez
de los importes, la estabilidad de los tipos de saldo ni la
correspondencia exacta de las rutas declaradas.

## 10. Identidad de las cuentas

### Objetivo
Comprobar si `account_uid` e `identification_hash` se mantienen
estables entre las extracciones de movimientos y saldos.

`account_index` identifica la posición de una cuenta en la extracción;
no se asumirá como identificador permanente.

### Comprobaciones
- Detectar identificadores nulos o vacíos.
- Contar las identidades observadas por banco e índice de cuenta.
- Comparar su presencia en ambos endpoints.

La estabilidad observada corresponde únicamente al periodo analizado.

In [0]:
# Leemos únicamente los identificadores y los metadatos.
account_observations_df = (
    spark.read
    .option("multiLine", "true")
    .json(all_transaction_paths + all_balance_paths)
    .select(
        "bank",
        "run_id",
        "endpoint",
        "account_uid",
        "identification_hash",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"/account=(\d+)/",
            1,
        ).cast("long").alias("account_index"),
    )
    .withColumn(
        "uid_valido",
        F.coalesce(
            F.length(F.trim("account_uid")) > 0,
            F.lit(False),
        ),
    )
    .withColumn(
        "hash_valido",
        F.coalesce(
            F.length(F.trim("identification_hash")) > 0,
            F.lit(False),
        ),
    )
)

# Resumen sin mostrar los identificadores.
display(
    account_observations_df
    .groupBy("bank", "account_index", "endpoint")
    .agg(
        F.countDistinct("run_id").alias("ejecuciones"),
        F.countDistinct(
            F.when(F.col("uid_valido"), F.col("account_uid"))
        ).alias("uids_distintos"),
        F.countDistinct(
            F.when(
                F.col("hash_valido"),
                F.col("identification_hash"),
            )
        ).alias("hashes_distintos"),
        F.sum(
            F.when(~F.col("uid_valido"), 1).otherwise(0)
        ).alias("archivos_sin_uid"),
        F.sum(
            F.when(~F.col("hash_valido"), 1).otherwise(0)
        ).alias("archivos_sin_hash"),
    )
    .orderBy("bank", "account_index", "endpoint")
)

# Relación entre ambos identificadores, sin mostrar sus valores.
# Si un banco tiene una sola pareja, ambos endpoints usan la misma
# identidad durante el periodo observado.
display(
    account_observations_df
    .where(F.col("uid_valido") & F.col("hash_valido"))
    .select(
        "bank",
        "account_index",
        "account_uid",
        "identification_hash",
    )
    .distinct()
    .groupBy("bank", "account_index")
    .agg(F.count("*").alias("parejas_uid_hash"))
    .orderBy("bank", "account_index")
)

### Conclusiones

Para cada banco se observa:
- Un único `account_uid` y un único `identification_hash`.
- Una única pareja de ambos identificadores compartida entre endpoints.
- Ningún archivo con estos identificadores nulos o vacíos.

Por tanto, no se observan cambios de identidad de cuenta durante
el periodo analizado.

Conservaremos ambos identificadores para mantener la trazabilidad.
`account_index` se conservará como metadato de la extracción,
sin utilizarlo como clave permanente.

Estos resultados no garantizan que los identificadores permanezcan
iguales después de renovar el consentimiento o reconectar una cuenta.

## 11. Identificadores de movimientos

### Objetivo
Evaluar si `transaction_id` permite identificar movimientos dentro
de cada banco y cuenta.

### Comprobaciones
- Identificadores nulos o vacíos.
- Identificadores distintos en el conjunto de extracciones.
- Identificadores repetidos dentro de una misma ejecución.

La repetición entre ejecuciones es esperable por las ventanas
solapadas. Todavía no se eliminará ningún registro.

In [0]:
transaction_observations_df = (
    spark.read
    .option("multiLine", "true")
    .json(all_transaction_paths)
    .select(
        "bank",
        "account_uid",
        "run_id",
        "extracted_at",
        F.explode("response.transactions").alias("transaction"),
    )
    .withColumn(
        "id_valido",
        F.coalesce(
            F.length(
                F.trim(F.col("transaction.transaction_id"))
            ) > 0,
            F.lit(False),
        ),
    )
)

# Cobertura de transaction_id.
display(
    transaction_observations_df
    .groupBy("bank", "account_uid")
    .agg(
        F.count("*").alias("observaciones"),
        F.sum(
            F.when(~F.col("id_valido"), 1).otherwise(0)
        ).alias("observaciones_sin_id"),
        F.countDistinct(
            F.when(
                F.col("id_valido"),
                F.col("transaction.transaction_id"),
            )
        ).alias("ids_distintos"),
    )
    .drop("account_uid")
    .orderBy("bank")
)

# ¿Se repite un mismo ID dentro de una ejecución y cuenta?
repeated_ids_per_run_df = (
    transaction_observations_df
    .where(F.col("id_valido"))
    .groupBy(
        "bank",
        "account_uid",
        "run_id",
        F.col("transaction.transaction_id").alias("transaction_id"),
    )
    .agg(F.count("*").alias("apariciones"))
    .where(F.col("apariciones") > 1)
)

display(
    repeated_ids_per_run_df
    .groupBy("bank")
    .agg(
        F.count("*").alias("grupos_id_repetidos"),
        F.max("apariciones").alias("max_apariciones_por_id"),
    )
    .orderBy("bank")
)

### Conclusiones

| Banco | Observaciones | Sin transaction_id | IDs distintos |
|---|---:|---:|---:|
| Laboral | 78 | 78 | 0 |
| Santander | 437 | 437 | 0 |

`transaction_id` no está informado en ninguna observación del
periodo analizado y no puede utilizarse como clave de movimiento.

La ausencia de resultados en el control de IDs repetidos se debe
a que todos los registros quedan excluidos por carecer de ID.

Antes de definir una clave alternativa, examinaremos la cobertura
de otras referencias proporcionadas por el banco. No eliminaremos
observaciones basándonos únicamente en fecha, importe y descripción:
pueden existir movimientos legítimos con esos mismos valores.

In [0]:
reference_coverage_df = (
    transaction_observations_df
    .select(
        "bank",
        "account_uid",
        F.explode(
            F.array(
                F.struct(
                    F.lit("entry_reference").alias("campo"),
                    F.col("transaction.entry_reference").alias("valor"),
                ),
                F.struct(
                    F.lit("reference_number").alias("campo"),
                    F.col("transaction.reference_number").alias("valor"),
                ),
            )
        ).alias("reference"),
    )
    .select(
        "bank",
        "account_uid",
        "reference.campo",
        "reference.valor",
    )
    .withColumn(
        "informado",
        F.coalesce(
            F.length(F.trim("valor")) > 0,
            F.lit(False),
        ),
    )
)

display(
    reference_coverage_df
    .groupBy("bank", "account_uid", "campo")
    .agg(
        F.count("*").alias("observaciones"),
        F.sum(
            F.when(F.col("informado"), 1).otherwise(0)
        ).alias("con_referencia"),
        F.countDistinct(
            F.when(F.col("informado"), F.col("valor"))
        ).alias("referencias_distintas"),
    )
    .drop("account_uid")
    .orderBy("bank", "campo")
)

### Cobertura de referencias alternativas

| Banco | Campo | Observaciones informadas | Referencias distintas |
|---|---|---:|---:|
| Laboral | entry_reference | 0 de 78 | 0 |
| Laboral | reference_number | 0 de 78 | 0 |
| Santander | entry_reference | 407 de 437 | 177 |
| Santander | reference_number | 0 de 437 | 0 |

`entry_reference` es una posible clave para parte de los movimientos
de Santander, pendiente de comprobar sus repeticiones y estabilidad.

Las 177 referencias distintas no equivalen todavía a 177 movimientos
únicos demostrados.

Quedan sin esta referencia las 78 observaciones de Laboral y
30 de Santander. No existe, por ahora, una clave bancaria informada
que cubra todo el conjunto.

In [0]:
display(
    transaction_observations_df
    .withColumn(
        "tiene_entry_reference",
        F.coalesce(
            F.length(
                F.trim(F.col("transaction.entry_reference"))
            ) > 0,
            F.lit(False),
        ),
    )
    .groupBy(
        "bank",
        F.col("transaction.status").alias("estado"),
        "tiene_entry_reference",
    )
    .agg(
        F.count("*").alias("observaciones"),
        F.countDistinct("run_id").alias("ejecuciones"),
        F.min("transaction.booking_date").alias("primera_booking_date"),
        F.max("transaction.booking_date").alias("ultima_booking_date"),
    )
    .orderBy("bank", "estado", "tiene_entry_reference")
)

### Referencias y estado del movimiento

Todas las observaciones analizadas tienen estado `BOOK`.

- Laboral: 78 observaciones sin `entry_reference`, en 19 ejecuciones.
- Santander: 30 observaciones sin referencia, en 15 ejecuciones,
  y 407 con referencia, en 19 ejecuciones.

La ausencia de referencia no se limita a movimientos pendientes.
No podemos descartar esos registros ni identificar movimientos
únicos contando únicamente las referencias informadas.

La estrategia de deduplicación continúa pendiente de definición.
Bronze deberá conservar las observaciones y su procedencia.

In [0]:
entry_reference_counts_df = (
    transaction_observations_df
    .where(
        F.length(
            F.trim(F.col("transaction.entry_reference"))
        ) > 0
    )
    .groupBy(
        "bank",
        "account_uid",
        "run_id",
        F.col("transaction.entry_reference").alias("entry_reference"),
    )
    .agg(F.count("*").alias("apariciones"))
)

display(
    entry_reference_counts_df
    .groupBy("bank")
    .agg(
        F.count("*").alias("grupos_ejecucion_referencia"),
        F.sum(
            F.when(F.col("apariciones") > 1, 1).otherwise(0)
        ).alias("grupos_con_repeticiones"),
        F.max("apariciones").alias("max_apariciones"),
    )
)

### Unicidad de entry_reference dentro de cada ejecución

En Santander se observan 407 combinaciones de ejecución, cuenta
y referencia. Ninguna contiene más de un registro.

`entry_reference` es una candidata a identificar movimientos
cuando está informada. Falta comprobar si una misma referencia
mantiene datos coherentes entre extracciones.

Este resultado no resuelve los registros sin referencia:
78 observaciones de Laboral y 30 de Santander.

## 12. Consistencia de las referencias entre extracciones

### Objetivo
Comprobar si las observaciones de una misma cuenta y `entry_reference`
mantienen el importe, la moneda, el sentido del movimiento y las fechas.

Una variación no implica necesariamente un error: puede representar
una actualización del banco. Si aparece, se analizará antes de
establecer la deduplicación.

In [0]:
reference_consistency_df = (
    transaction_observations_df
    .where(
        F.length(
            F.trim(F.col("transaction.entry_reference"))
        ) > 0
    )
    .groupBy(
        "bank",
        "account_uid",
        F.col("transaction.entry_reference").alias("entry_reference"),
    )
    .agg(
        F.count("*").alias("observaciones"),
        F.countDistinct("run_id").alias("ejecuciones"),
        F.countDistinct(
            F.to_json(
                F.struct(
                    F.col("transaction.transaction_amount.amount")
                    .alias("importe"),
                    F.col("transaction.transaction_amount.currency")
                    .alias("moneda"),
                    F.col("transaction.credit_debit_indicator")
                    .alias("sentido"),
                    F.col("transaction.booking_date")
                    .alias("fecha_contable"),
                    F.col("transaction.transaction_date")
                    .alias("fecha_operacion"),
                    F.col("transaction.value_date")
                    .alias("fecha_valor"),
                ),
                options={"ignoreNullFields": "false"},
            )
        ).alias("versiones_campos_comparados"),
    )
)

display(
    reference_consistency_df
    .groupBy("bank")
    .agg(
        F.count("*").alias("referencias"),
        F.sum(
            F.when(F.col("ejecuciones") > 1, 1).otherwise(0)
        ).alias("referencias_en_varias_ejecuciones"),
        F.sum(
            F.when(
                F.col("versiones_campos_comparados") > 1, 1
            ).otherwise(0)
        ).alias("referencias_con_cambios"),
        F.max("versiones_campos_comparados")
        .alias("max_versiones"),
    )
)

### Conclusiones

Se identifican 177 referencias distintas de Santander:
- 41 aparecen en más de una ejecución.
- Ninguna presenta cambios en importe, moneda, sentido del movimiento,
  fecha contable, fecha de operación o fecha valor.

Junto con la ausencia de referencias repetidas dentro de una ejecución,
esto respalda usar `(bank, account_uid, entry_reference)` como clave
candidata cuando la referencia esté informada.

La conclusión se limita al periodo y a los campos analizados.
No demuestra que el resto del contenido permanezca igual.

Sigue pendiente identificar los movimientos sin referencia:
78 observaciones de Laboral y 30 de Santander.

## 13. Repeticiones de movimientos sin referencia

### Objetivo
Comparar el contenido completo de los movimientos sin `entry_reference`.

Se calculará una huella del contenido del movimiento, excluyendo
los metadatos de extracción, como `run_id` y `extracted_at`.

La huella permite detectar contenidos iguales, pero no demuestra
que correspondan al mismo movimiento real: podrían existir dos
operaciones legítimas idénticas.

In [0]:
unreferenced_df = (
    transaction_observations_df
    .where(
        F.coalesce(
            F.length(
                F.trim(F.col("transaction.entry_reference"))
            ),
            F.lit(0),
        ) == 0
    )
    .withColumn(
        "content_hash",
        F.sha2(
            F.to_json(
                F.col("transaction"),
                options={"ignoreNullFields": "false"},
            ),
            256,
        ),
    )
)

# Apariciones del mismo contenido dentro de cada ejecución y cuenta.
content_per_run_df = (
    unreferenced_df
    .groupBy("bank", "account_uid", "run_id", "content_hash")
    .agg(F.count("*").alias("apariciones"))
)

# Presencia del mismo contenido entre ejecuciones.
content_across_runs_df = (
    content_per_run_df
    .groupBy("bank", "account_uid", "content_hash")
    .agg(
        F.sum("apariciones").alias("observaciones"),
        F.countDistinct("run_id").alias("ejecuciones"),
        F.max("apariciones").alias("max_apariciones_en_una_ejecucion"),
    )
)

display(
    content_across_runs_df
    .groupBy("bank")
    .agg(
        F.sum("observaciones").alias("observaciones_sin_referencia"),
        F.count("*").alias("contenidos_distintos"),
        F.sum(
            F.when(F.col("ejecuciones") > 1, 1).otherwise(0)
        ).alias("contenidos_en_varias_ejecuciones"),
        F.sum(
            F.when(
                F.col("max_apariciones_en_una_ejecucion") > 1,
                1,
            ).otherwise(0)
        ).alias("contenidos_repetidos_dentro_de_una_ejecucion"),
        F.max("max_apariciones_en_una_ejecucion")
        .alias("max_apariciones_en_una_ejecucion"),
    )
    .orderBy("bank")
)

### Conclusiones

En Laboral se observan 45 contenidos distintos entre 78 observaciones
sin referencia. Seis contenidos aparecen en varias ejecuciones.

En Santander, las 30 observaciones sin referencia presentan contenidos
distintos entre sí.

No se detectan contenidos idénticos repetidos dentro de una misma
ejecución y cuenta.

La huella del contenido identifica repeticiones exactas, pero no
constituye todavía una clave de movimiento:
- Una actualización puede generar una huella diferente.
- Un movimiento inicialmente sin referencia podría recibirla después.
- Dos operaciones reales podrían compartir el mismo contenido.

Por tanto, estos recuentos no se interpretan como movimientos únicos.